In [1]:
!pip install -q fastapi uvicorn sqlalchemy pydantic httpx nest-asyncio

In [2]:
import json
from typing import List, Optional
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from sqlalchemy import create_engine, Column, Integer, String, Boolean, ForeignKey
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, relationship, Session

# We will use SQLite because its a super easy database that saves into a local file
# No need for complex database servers!
DATABASE_URL = "sqlite:///./quiz_database.db"

# connect_args is needed for sqlite to play nice with FastAPI's threads
engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

# --- DATABASE MODELS (How our tables look inside the DB) ---

class Question(Base):
    __tablename__ = "questions"

    id = Column(Integer, primary_key=True, index=True)
    question_text = Column(String, nullable=False)
    category = Column(String, default="General")

    # This cascade delete is super important! If we delete a question,
    # we don't want orphan choices floating around in the database.
    choices = relationship("Choice", back_populates="parent_question", cascade="all, delete-orphan")


class Choice(Base):
    __tablename__ = "choices"

    id = Column(Integer, primary_key=True, index=True)
    choice_text = Column(String, nullable=False)
    is_correct = Column(Boolean, default=False)
    question_id = Column(Integer, ForeignKey("questions.id", ondelete="CASCADE"), nullable=False)

    # Connecting back to the original question
    parent_question = relationship("Question", back_populates="choices")


# Create the tables! If they are already there, SQLite will just ignore this.
Base.metadata.create_all(bind=engine)
print("Database and tables are setup and ready")

Database and tables are setup and ready


/tmp/ipykernel_733/584794928.py:17: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [3]:
from pydantic import BaseModel, Field
from typing import List, Optional

# These are Pydantic schemas to validate incoming data from the user
# It acts like a guard dog, making sure nobody sends bad data!

class ChoiceBase(BaseModel):
    choice_text: str = Field(..., min_length=1, description="Text for the multiple choice answer")
    is_correct: bool = False

class ChoiceCreate(ChoiceBase):
    question_id: int

class ChoiceResponse(ChoiceBase):
    id: int
    question_id: int

    class Config:
        from_attributes = True  # This lets Pydantic read SQLAlchemy models directly!


class QuestionBase(BaseModel):
    question_text: str = Field(..., min_length=5, description="The quiz question text")
    category: Optional[str] = "General"

class QuestionCreate(QuestionBase):
    pass

class QuestionResponse(QuestionBase):
    id: int
    # This nesting automatically displays all associated choices when we fetch a question!
    choices: List[ChoiceResponse] = []

    class Config:
        from_attributes = True

/tmp/ipykernel_733/2308844572.py:14: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class ChoiceResponse(ChoiceBase):
/tmp/ipykernel_733/2308844572.py:29: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class QuestionResponse(QuestionBase):


In [5]:
class QuestionResponse(QuestionBase):
    id: int
    # This nesting automatically displays all associated choices when we fetch a question!
    choices: List[ChoiceResponse] = []

    class Config:
        from_attributes = True


# let's set up the fastapi app objct here
app = FastAPI(title="Quiz Backend API", description="CRUD API for managing questions and choises")

# dependancy to get the db sesion. this is super importnt for making sure we close conection!
def get_db():
    my_db = SessionLocal()
    try:
        yield my_db
    finally:
        my_db.close() # always close to prevent leaking connections!


# --- QUESTION ENDPOINTS ---

# Create a brand new question
@app.post("/questions", response_model=QuestionResponse, status_code=status.HTTP_201_CREATED)
def create_question(question: QuestionCreate, db: Session = Depends(get_db)):
    # create database record
    db_question = Question(
        question_text=question.question_text,
        category=question.category
    )
    db.add(db_question)
    db.commit() # save to sqlite
    db.refresh(db_question) # get the generated ID
    return db_question

# Get all questions with choices nested inside
@app.get("/questions", response_model=List[QuestionResponse])
def read_questions(skip: int = 0, limit: int = 100, db: Session = Depends(get_db)):
    questions = db.query(Question).offset(skip).limit(limit).all()
    return questions

# Get a specific question by ID
@app.get("/questions/{id}", response_model=QuestionResponse)
def read_question(id: int, db: Session = Depends(get_db)):
    question = db.query(Question).filter(Question.id == id).first()
    if not question:
        raise HTTPException(status_code=404, detail="oops! question not found")
    return question

# Update an exisiting question
@app.put("/questions/{id}", response_model=QuestionResponse)
def update_question(id: int, updated_question: QuestionCreate, db: Session = Depends(get_db)):
    db_question = db.query(Question).filter(Question.id == id).first()
    if not db_question:
        raise HTTPException(status_code=404, detail="queston to update does not exist")

    db_question.question_text = updated_question.question_text
    db_question.category = updated_question.category
    db.commit()
    db.refresh(db_question)
    return db_question

# Delete question and cascade delete choices automatically
@app.delete("/questions/{id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_question(id: int, db: Session = Depends(get_db)):
    db_question = db.query(Question).filter(Question.id == id).first()
    if not db_question:
        raise HTTPException(status_code=404, detail="can't delete, question not found")

    db.delete(db_question) # sqlalchemy handles deleting choices due to cascade delete!
    db.commit()
    return None # 204 no content returns nothing


# --- CHOICE ENDPOINTS ---

# Add a brand new choice to an exisiting question
@app.post("/choices", response_model=ChoiceResponse, status_code=status.HTTP_201_CREATED)
def create_choice(choice: ChoiceCreate, db: Session = Depends(get_db)):
    # first verify if the question realy exists
    question_exists = db.query(Question).filter(Question.id == choice.question_id).first()
    if not question_exists:
        raise HTTPException(status_code=400, detail="cant attach choice to non-existing question")

    db_choice = Choice(
        choice_text=choice.choice_text,
        is_correct=choice.is_correct,
        question_id=choice.question_id
    )
    db.add(db_choice)
    db.commit()
    db.refresh(db_choice)
    return db_choice

# Fetch all choices (general debug/audit endpoint)
@app.get("/choices", response_model=List[ChoiceResponse])
def read_choices(db: Session = Depends(get_db)):
    return db.query(Choice).all()

# Update a choice (e.g. modify answer text or corectness)
@app.put("/choices/{id}", response_model=ChoiceResponse)
def update_choice(id: int, updated_choice: ChoiceCreate, db: Session = Depends(get_db)):
    db_choice = db.query(Choice).filter(Choice.id == id).first()
    if not db_choice:
        raise HTTPException(status_code=404, detail="choice not found to update")

    db_choice.choice_text = updated_choice.choice_text
    db_choice.is_correct = updated_choice.is_correct
    db_choice.question_id = updated_choice.question_id
    db.commit()
    db.refresh(db_choice)
    return db_choice

# Delete a single choice safely
@app.delete("/choices/{id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_choice(id: int, db: Session = Depends(get_db)):
    db_choice = db.query(Choice).filter(Choice.id == id).first()
    if not db_choice:
        raise HTTPException(status_code=404, detail="choice to delete not found")
    db.delete(db_choice)
    db.commit()
    return None

/tmp/ipykernel_733/1939077530.py:1: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class QuestionResponse(QuestionBase):


In [6]:
class QuestionResponse(QuestionBase):
    id: int
    # This nesting automatically displays all associated choices when we fetch a question!
    choices: List[ChoiceResponse] = []

    class Config:
        from_attributes = True


# let's set up the fastapi app objct here
app = FastAPI(title="Quiz Backend API", description="CRUD API for managing questions and choises")

# dependancy to get the db sesion. this is super importnt for making sure we close conection!
def get_db():
    my_db = SessionLocal()
    try:
        yield my_db
    finally:
        my_db.close() # always close to prevent leaking connections!


#  QUESTION ENDPOINTS

# Create a brand new question
@app.post("/questions", response_model=QuestionResponse, status_code=status.HTTP_201_CREATED)
def create_question(question: QuestionCreate, db: Session = Depends(get_db)):
    # create database record
    db_question = Question(
        question_text=question.question_text,
        category=question.category
    )
    db.add(db_question)
    db.commit() # save to sqlite
    db.refresh(db_question) # get the generated ID
    return db_question

# Get all questions with choices nested inside
@app.get("/questions", response_model=List[QuestionResponse])
def read_questions(skip: int = 0, limit: int = 100, db: Session = Depends(get_db)):
    questions = db.query(Question).offset(skip).limit(limit).all()
    return questions

# Get a specific question by ID
@app.get("/questions/{id}", response_model=QuestionResponse)
def read_question(id: int, db: Session = Depends(get_db)):
    question = db.query(Question).filter(Question.id == id).first()
    if not question:
        raise HTTPException(status_code=404, detail="oops! question not found")
    return question

# Update an exisiting question
@app.put("/questions/{id}", response_model=QuestionResponse)
def update_question(id: int, updated_question: QuestionCreate, db: Session = Depends(get_db)):
    db_question = db.query(Question).filter(Question.id == id).first()
    if not db_question:
        raise HTTPException(status_code=404, detail="queston to update does not exist")

    db_question.question_text = updated_question.question_text
    db_question.category = updated_question.category
    db.commit()
    db.refresh(db_question)
    return db_question

# Delete question and cascade delete choices automatically
@app.delete("/questions/{id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_question(id: int, db: Session = Depends(get_db)):
    db_question = db.query(Question).filter(Question.id == id).first()
    if not db_question:
        raise HTTPException(status_code=404, detail="can't delete, question not found")

    db.delete(db_question) # sqlalchemy handles deleting choices due to cascade delete!
    db.commit()
    return None # 204 no content returns nothing


# CHOICE ENDPOINTS

# Add a brand new choice to an exisiting question
@app.post("/choices", response_model=ChoiceResponse, status_code=status.HTTP_201_CREATED)
def create_choice(choice: ChoiceCreate, db: Session = Depends(get_db)):
    # first verify if the question realy exists
    question_exists = db.query(Question).filter(Question.id == choice.question_id).first()
    if not question_exists:
        raise HTTPException(status_code=400, detail="cant attach choice to non-existing question")

    db_choice = Choice(
        choice_text=choice.choice_text,
        is_correct=choice.is_correct,
        question_id=choice.question_id
    )
    db.add(db_choice)
    db.commit()
    db.refresh(db_choice)
    return db_choice

# Fetch all choices (general debug/audit endpoint)
@app.get("/choices", response_model=List[ChoiceResponse])
def read_choices(db: Session = Depends(get_db)):
    return db.query(Choice).all()

# Update a choice (e.g. modify answer text or corectness)
@app.put("/choices/{id}", response_model=ChoiceResponse)
def update_choice(id: int, updated_choice: ChoiceCreate, db: Session = Depends(get_db)):
    db_choice = db.query(Choice).filter(Choice.id == id).first()
    if not db_choice:
        raise HTTPException(status_code=404, detail="choice not found to update")

    db_choice.choice_text = updated_choice.choice_text
    db_choice.is_correct = updated_choice.is_correct
    db_choice.question_id = updated_choice.question_id
    db.commit()
    db.refresh(db_choice)
    return db_choice

# Delete a single choice safely
@app.delete("/choices/{id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_choice(id: int, db: Session = Depends(get_db)):
    db_choice = db.query(Choice).filter(Choice.id == id).first()
    if not db_choice:
        raise HTTPException(status_code=404, detail="choice to delete not found")
    db.delete(db_choice)
    db.commit()
    return None


#  TESTING AND DATA SEEDING SUITE
# This runs automaticaly when you execute this block in colab!
# It acts like a live simulator to veriify everything is working great

if __name__ == "__main__":
    from fastapi.testclient import TestClient

    # first, let's create the tables in our SQLite db!
    Base.metadata.create_all(bind=engine)
    print("Tables created in memory/local SQLite database! 🎉")

    client = TestClient(app)

    # 1. Let's create a quiz question
    print("\n--- Test 1: Creating a Question ---")
    q_data = {
        "question_text": "What does SQL stand for in database systems?",
        "category": "Databases"
    }
    res_q = client.post("/questions", json=q_data)
    assert res_q.status_code == 201
    q_json = res_q.json()
    q_id = q_json["id"]
    print(f"Success! Created question ID {q_id}: '{q_json['question_text']}'")

    # 2. Add some multiple choises to this question
    print("\n--- Test 2: Adding Choices ---")
    choices_data = [
        {"choice_text": "Structured Query Language", "is_correct": True, "question_id": q_id},
        {"choice_text": "Simple Query Line", "is_correct": False, "question_id": q_id},
        {"choice_text": "Strong Quality Log", "is_correct": False, "question_id": q_id}
    ]
    for c in choices_data:
        res_c = client.post("/choices", json=c)
        assert res_c.status_code == 201
        print(f"Created choice: '{res_c.json()['choice_text']}' (Correct: {res_c.json()['is_correct']})")

    # 3. Fetch all questions and check if choices are nested inside
    print("\n--- Test 3: Fetching Questions with Nested Choices ---")
    res_get = client.get("/questions")
    assert res_get.status_code == 200
    questions_list = res_get.json()
    print(f"Fetched {len(questions_list)} questions.")
    for question in questions_list:
        print(f"Question: {question['question_text']}")
        for choice in question['choices']:
            print(f"  - Option: {choice['choice_text']} (Is Correct: {choice['is_correct']})")

    # 4. Update the question category or text
    print("\n--- Test 4: Updating a Question ---")
    updated_data = {
        "question_text": "What does SQL stand for?",
        "category": "SQL Basics"
    }
    res_put = client.put(f"/questions/{q_id}", json=updated_data)
    assert res_put.status_code == 200
    print(f"Updated Question: {res_put.json()['question_text']} | Category: {res_put.json()['category']}")

    # 5. Delete question and verify choices are deleted by cascade automatically
    print("\n--- Test 5: Deleting Question (Verifying Cascade Delete) ---")
    # Let's count choices before delete
    res_all_choices_before = client.get("/choices")
    print(f"Total choices in DB before delete: {len(res_all_choices_before.json())}")

    res_del = client.delete(f"/questions/{q_id}")
    assert res_del.status_code == 204
    print(f"Deleted Question ID {q_id} succesfully!")

    # Verify all associated choices are completely gone
    res_all_choices_after = client.get("/choices")
    print(f"Total choices in DB after delete: {len(res_all_choices_after.json())}")
    assert len(res_all_choices_after.json()) == 0
    print("Success! Cascadeing delete worked perfectly. Choices are deleted automaticaly! ")

Tables created in memory/local SQLite database! 🎉

--- Test 1: Creating a Question ---
Success! Created question ID 1: 'What does SQL stand for in database systems?'

--- Test 2: Adding Choices ---
Created choice: 'Structured Query Language' (Correct: True)
Created choice: 'Simple Query Line' (Correct: False)
Created choice: 'Strong Quality Log' (Correct: False)

--- Test 3: Fetching Questions with Nested Choices ---
Fetched 1 questions.
Question: What does SQL stand for in database systems?
  - Option: Structured Query Language (Is Correct: True)
  - Option: Simple Query Line (Is Correct: False)
  - Option: Strong Quality Log (Is Correct: False)

--- Test 4: Updating a Question ---
Updated Question: What does SQL stand for? | Category: SQL Basics

--- Test 5: Deleting Question (Verifying Cascade Delete) ---
Total choices in DB before delete: 3
Deleted Question ID 1 succesfully!
Total choices in DB after delete: 0
Success! Cascadeing delete worked perfectly. Choices are deleted automat

/tmp/ipykernel_733/3366708618.py:1: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class QuestionResponse(QuestionBase):


I chose SQLite and FastAPI because it lets me run everything locally in Colab without configuring a separate database server. I set up cascade deletes on the relationship so SQLite automatically cleans up orphan answer choices when a question is deleted. Lastly, Pydantic schemas validate the input text to prevent corrupt data from hitting our tables.